In [514]:
import tensorflow as tf
import tensorflow_probability as tfp
import time
import numpy as np

# For convenience
tfd = tfp.distributions
tfb = tfp.bijectors
DTYPE=np.float32

In [515]:
class RegularGridInterpolator:
    """
    A reusable nD linear interpolator.
    
    Initializes once with the reference grid and values. The object can then be
    called repeatedly with new query points for fast, graph-based interpolation.
    """
    def __init__(self, x_ref_min, x_ref_max, y_ref):
        # --- ONE-TIME SETUP ---
        # Store grid data as TensorFlow constants to embed them in the graph.
        self.x_ref_min = tf.constant(x_ref_min, dtype=tf.float32)
        self.x_ref_max = tf.constant(x_ref_max, dtype=tf.float32)
        self.y_ref = tf.constant(y_ref, dtype=tf.float32)

    @tf.function
    def __call__(self, x_query):
        """
        Performs interpolation. This method is compiled into a static graph
        by @tf.function for high performance.
        
        Args:
            x_query: A tensor of x-coordinates to interpolate.
        
        Returns:
            A tensor of interpolated y-values.
        """
        # --- REPEATED, FAST EXECUTION ---
        return tfp.math.batch_interp_regular_nd_grid(
            x=x_query, 
            x_ref_min=self.x_ref_min, 
            x_ref_max=self.x_ref_max, 
            y_ref=self.y_ref,
            axis=-3
        )

# --- Usage ---

# 1. DEFINE YOUR GRID DATA (Do this only once)
print("--- Setting up the interpolator ---")
y_values = np.loadtxt("Kl-ppiplus.dat")
y_values = np.reshape(y_values,(100,100,100))
x_min = [0, 0, 0]
x_max = [100,40,np.pi]

# 2. CREATE THE REUSABLE INTERPOLATOR OBJECT (Do this only once)
interpolator = RegularGridInterpolator(x_ref_min=x_min, x_ref_max=x_max, y_ref=y_values)
print("Interpolator created.")

# 3. USE THE INTERPOLATOR MANY TIMES (This part is fast)
print("\n--- Using the interpolator ---")

# First call (includes compilation time)
start_time = time.time()
query1 = tf.constant([[11.0, 15.0, 19.5]], dtype=tf.float32)
result1 = interpolator(query1)
print(f"First call took: {time.time() - start_time:.6f} seconds")
print(f"Result for {query1.numpy()}: {result1.numpy()}")

# Subsequent calls (much faster)
start_time = time.time()
query2 = tf.random.uniform(shape=[1,3], minval=x_min, maxval=x_max)
result2 = interpolator(query2)
print(f"\nSecond call took: {time.time() - start_time:.6f} seconds")
print(f"Result for random queries: {result2.numpy()}")

# Another call
start_time = time.time()
query3 = tf.constant([[12.5, 17.5,1]], dtype=tf.float32)
result3 = interpolator(query3)
print(f"\nThird call took: {time.time() - start_time:.6f} seconds")
print(f"Result for {query3.numpy()}: {result3.numpy()}")

--- Setting up the interpolator ---
Interpolator created.

--- Using the interpolator ---
First call took: 0.051617 seconds
Result for [[11.  15.  19.5]]: [0.7645925]

Second call took: 0.000596 seconds
Result for random queries: [0.95719457]

Third call took: 0.000285 seconds
Result for [[12.5 17.5  1. ]]: [0.97838974]


In [516]:

# 1. Define the parameters of your target correlated Gaussian distribution.
# The covariance matrix MUST be symmetric and positive semi-definite.
target_mean = tf.constant([4.0, 0.0, -0.5], dtype=tf.float32)
target_covariance = tf.constant([
    [80, 0.0, 0.0],
    [0.0, 30.0, -0.0],
    [0.0, -0.0, 30.0]
], dtype=tf.float32)

# The bijector is parameterized by the Cholesky factor of the covariance.
cholesky_factor = tf.linalg.cholesky(target_covariance)
affine_bijector = tfb.Chain([
    tfb.Shift(target_mean),
    tfb.ScaleMatvecTriL(scale_tril=cholesky_factor)
])


# 3. Define the base and transformed distributions.
base_distribution = tfd.MultivariateNormalDiag(loc=tf.zeros(3), scale_diag=tf.ones(3))

target_distribution = tfd.TransformedDistribution(
    distribution=base_distribution,
    bijector=affine_bijector
)
# --- Verification ---
# Let's test it by drawing a large number of samples.

num_samples = 200000
print(cholesky_factor)
# Sample from the final distribution.
# TFP handles the transformation automatically.
x_samples = target_distribution.sample(num_samples)

# Calculate the mean and covariance of the generated samples.
empirical_mean = tf.reduce_mean(x_samples, axis=0)
empirical_covariance = tfp.stats.covariance(x_samples)

print("--- Target Parameters ---")
print(f"Target Mean:\n{target_mean.numpy()}")
print(f"Target Covariance:\n{target_covariance.numpy()}")

print("\n--- Empirical Results from Samples ---")
print(f"Empirical Mean:\n{empirical_mean.numpy()}")
print(f"Empirical Covariance:\n{empirical_covariance.numpy()}")

tf.Tensor(
[[ 8.944272  0.        0.      ]
 [ 0.        5.477226  0.      ]
 [ 0.       -0.        5.477226]], shape=(3, 3), dtype=float32)
--- Target Parameters ---
Target Mean:
[ 4.   0.  -0.5]
Target Covariance:
[[80.  0.  0.]
 [ 0. 30. -0.]
 [ 0. -0. 30.]]

--- Empirical Results from Samples ---
Empirical Mean:
[ 3.981059    0.03766736 -0.5066819 ]
Empirical Covariance:
[[ 7.9964493e+01 -2.4638668e-01  1.7289580e-01]
 [-2.4638668e-01  3.0086298e+01  1.5055899e-02]
 [ 1.7289582e-01  1.5055906e-02  2.9862675e+01]]


In [517]:
import tensorflow as tf
import tensorflow_probability as tfp

tfb = tfp.bijectors
tfd = tfp.distributions
DTYPE = tf.float32

class Cholesky(tfb.Bijector):
    def __init__(self, event_shape, name='TrainableCholesky'):
        super(Cholesky, self).__init__(
            forward_min_event_ndims=1,
            validate_args=False,
            name=name)
        
        dim = event_shape
        # 1. Create a trainable vector of parameters for the lower-triangular matrix
        # For a DxD matrix, there are D*(D+1)/2 parameters.
        num_params = dim * (dim + 1) // 2
        self._cholesky_params = tf.Variable(
            tfp.math.fill_triangular_inverse(cholesky_factor),
            name='cholesky_params',
            dtype=DTYPE
        )
        self._shift = tf.Variable([7,0,-0.5],name='shift',dtype=DTYPE)

        
    def _get_transform(self):
        # 2. Construct the lower-triangular matrix from the parameters
        # This ensures the diagonal is positive, making it a valid Cholesky factor.
        return tfp.math.fill_triangular(self._cholesky_params)
        
    def _forward(self, x):
        # Forward pass: maps base distribution to target -> y = L @ x
        # Use matvec for proper batch handling: [B, D] -> [B, D]
        return tf.linalg.matvec(self._get_transform(), x)
    
    def _inverse(self, y):
        # Inverse pass: maps target back to base -> x = L_inv @ y
        return tf.linalg.triangular_solve(self._get_transform(), y[..., tf.newaxis])[..., 0]

    def _forward_log_det_jacobian(self, x):
        # log|det(L)| = sum(log(diag(L)))
        return tf.reduce_sum(tf.math.log(tf.linalg.diag_part(self._get_transform())))

    def _inverse_log_det_jacobian(self, y):
        # log|det(L_inv)| = -log|det(L)|
        return -tf.reduce_sum(tf.math.log(tf.linalg.diag_part(self._get_transform())))



        
affine_bijector = Cholesky(event_shape=3)


# 3. Define the base and transformed distributions.
base_distribution = tfd.MultivariateNormalDiag(loc=tf.zeros(3), scale_diag=tf.ones(3))

target_distribution = tfd.TransformedDistribution(
    distribution=base_distribution,
    bijector=affine_bijector
)
flow = tfd.TransformedDistribution(
    distribution=base_distribution,
    #we reverse the secuence because its actually applied first bijector in the list first otherwise its in a different direction
    bijector=affine_bijector
)

In [518]:

@tf.function
def KL_Divergence(xs,true_ys,qs):
    r=tf.norm(xs, ord='euclidean',axis=1)
    q=tf.norm(qs,ord='euclidean',axis=1)
    rqs=tf.tensordot(q,r,axes=0)
    trqs=tf.tensordot(qs,xs,axes=[[1],[1]])
    angle=tf.math.acos(trqs/rqs)
    trs,tqs=tf.meshgrid(q,r,indexing="ij")
    #trs=tf.reshape(trs, [-1])
    #tqs=tf.reshape(tqs, [-1])
    #angle=tf.reshape(angle, [-1])
    trs=tf.transpose(trs)
    tqs=tf.transpose(tqs)
    angle=tf.transpose(angle)
    ys=tf.transpose((tf.stack([tqs,trs,angle])))
    ys=interpolator(ys)
    ys=tf.math.reduce_mean(ys,axis=1)
    return tf.norm(ys-true_ys,ord='euclidean')

In [519]:
@tf.function
def compute_loss(xs, true_ys, qs, interpolator):
    # 1. Calculate vector magnitudes
    r = tf.norm(xs, ord='euclidean', axis=1)
    q = tf.norm(qs, ord='euclidean', axis=1)

    # 2. Calculate pairwise dot products and magnitudes
    dot_products = tf.tensordot(qs, xs, axes=[[1], [1]]) # Shape: [M, N]
    mag_products = tf.tensordot(q, r, axes=0)           # Shape: [M, N]

    # 3. Calculate angles with improved numerical stability
    ratio = dot_products / mag_products
    clipped_ratio = tf.clip_by_value(ratio, -1.0, 1.0)
    angle = tf.math.acos(clipped_ratio)                 # Shape: [M, N]

    # 4. Create coordinate grids
    q_grid, r_grid = tf.meshgrid(q, r, indexing="ij")   # Shapes: [M, N]

    # 5. Construct coordinate tensor directly (no transposing)
    # Shape becomes [M, N, 3]
    coords = tf.stack([q_grid, r_grid, angle], axis=-1)

    # 6. Interpolate and average results
    interpolated_ys = interpolator(coords)
    # Average over the N-dimension
    mean_ys = tf.math.reduce_mean(interpolated_ys, axis=1) # Shape: [M,]

    # 7. Return the final loss
    return tf.norm(mean_ys - true_ys, ord='euclidean')

In [520]:
qs=np.zeros(shape=(8000,3),dtype=np.float32)
true_Cq=np.loadtxt("65.dat",dtype=np.float32)
q_max=50
Nqs=20
for i in range(Nqs):
    for j in range(Nqs):
        for k in range(Nqs):
            qs[k+Nqs*j+Nqs*Nqs*i][0]=-q_max+2*q_max/Nqs*i
            qs[k+Nqs*j+Nqs*Nqs*i][1]=-q_max+2*q_max/Nqs*j
            qs[k+Nqs*j+Nqs*Nqs*i][2]=-q_max+2*q_max/Nqs*k


In [539]:
# Training loop
n_epochs = 10
n_iter = 1000
n_samples = 1000
from tensorflow.keras.utils import Progbar                         

  
#these are optimizer and params
lr_decay = .1
learning_rate = .001
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

for epoch in range(n_epochs):
    print('Epoch {:}/{:}'.format(epoch, n_epochs))
    progbar = Progbar(n_iter)
    for iter in range(n_iter):
        

        with tf.GradientTape() as ae_tape:
              
            xs_m = flow.sample(n_samples)
            loss = compute_loss(xs_m,true_Cq[::8],qs[::8],interpolator)
          
        gradients = ae_tape.gradient(loss, flow.trainable_variables)
        optimizer.apply_gradients(zip(gradients, flow.trainable_variables))
        progbar.add(1, values=[('loss', loss)])

Epoch 0/10
   3/1000 ━━━━━━━━━━━━━━━━━━━━ 6:06:27 22s/step - loss: 0.3537

KeyboardInterrupt: 

In [533]:
df=flow.sample(10000)

In [538]:
np.cov(np.transpose(df))

array([[80.00085421,  0.48306037,  0.84707076],
       [ 0.48306037, 30.00115359, -0.17329067],
       [ 0.84707076, -0.17329067, 30.14542468]])